In [ ]:
import pandas as pd

# load just the datasets q01 needs.
factor = 1
DATA_ROOT = f"/home/dias-benchmarks/tpch/data/factor_{factor}"
STORAGE_OPTS = {}  # or load from JSON

def load_lineitem(
    data_folder: str, **storage_options
) -> pd.DataFrame:
    data_path = data_folder + "/lineitem.tbl"
    # df = pd.read_parquet(
    #     data_path,  storage_options=storage_options
    # )
    df = pd.read_csv(data_path, sep="|")
    return df
lineitem = load_lineitem(DATA_ROOT, **STORAGE_OPTS)

In [ ]:
def load_orders(
    data_folder: str, **storage_options
) -> pd.DataFrame:
    data_path = data_folder + "/orders.tbl"
    # df = pd.read_parquet(
    #     data_path,  storage_options=storage_options
    # )
    df = pd.read_csv(data_path, sep="|")
    return df
orders = load_orders(DATA_ROOT, **STORAGE_OPTS)

In [ ]:
def load_supplier(
    data_folder: str, **storage_options
) -> pd.DataFrame:
    data_path = data_folder + "/supplier.tbl"
    # df = pd.read_parquet(
    #     data_path,  storage_options=storage_options
    # )
    df = pd.read_csv(data_path, sep="|")
    return df
supplier = load_supplier(DATA_ROOT, **STORAGE_OPTS)

In [ ]:
def load_nation(
    data_folder: str, **storage_options
) -> pd.DataFrame:
    data_path = data_folder + "/nation.tbl"
    # df = pd.read_parquet(
    #     data_path,  storage_options=storage_options
    # )
    df = pd.read_csv(data_path, sep="|")
    return df
nation = load_nation(DATA_ROOT, **STORAGE_OPTS)

In [ ]:
### cell 0 ###

# Filter lineitems where receipt date > commit date
mask_date = lineitem.L_RECEIPTDATE > lineitem.L_COMMITDATE
fi = lineitem[mask_date]

# Compute unique supplier counts per order
orig_count = (
    lineitem
    .groupby('L_ORDERKEY', sort=False)
    .agg({'L_SUPPKEY': 'nunique'})
    .reset_index()
    .rename(columns={'L_SUPPKEY': 'orig_count'})
)
after_count = (
    fi
    .groupby('L_ORDERKEY', sort=False)
    .agg({'L_SUPPKEY': 'nunique'})
    .reset_index()
    .rename(columns={'L_SUPPKEY': 'after_count'})
)

# Identify valid orders (orig_count >1 and after_count ==1)
valid_orders = orig_count.merge(after_count, on='L_ORDERKEY', how='inner')
valid_orders = valid_orders[(valid_orders.orig_count > 1) & (valid_orders.after_count == 1)]['L_ORDERKEY']

# Restrict to orders with status 'F'
f_orders = orders[orders.O_ORDERSTATUS == 'F']['O_ORDERKEY']
valid_orders = valid_orders[valid_orders.isin(f_orders)]

# Pre-filter Saudi Arabian suppliers
supplier_sa = supplier.merge(
    nation[nation.N_NAME == 'SAUDI ARABIA'][['N_NATIONKEY']],
    left_on='S_NATIONKEY',
    right_on='N_NATIONKEY',
    how='inner'
)[['S_SUPPKEY', 'S_NAME']]

# Final aggregation: count waiting lineitems per Saudi supplier
total = (
    lineitem[mask_date & lineitem.L_ORDERKEY.isin(valid_orders)]
    .merge(supplier_sa, left_on='L_SUPPKEY', right_on='S_SUPPKEY', how='inner')
    .groupby('S_NAME', as_index=False, sort=False)
    .size()
    .rename(columns={'size': 'NUMWAIT'})
    .sort_values(['NUMWAIT', 'S_NAME'], ascending=[False, True])
)